# 원하는 포즈로 이미지 만드는 도구 (Pose → Image)

전신 OpenPose 스켈레톤을 **ControlNet 조건**으로 넣어, **Stable Diffusion 1.5**가 그 자세의 **전신 서있는 인물**을 생성한다. 프롬프트만 바꿔도 자세는 유지된다.

> GPU(Colab T4)면 빠르고, CPU면 느리지만 동작. **seed 고정**은 같은 실행 환경에서 결과의 반복성을 높이지만, 장치·정밀도·라이브러리 버전이 다르면 픽셀 단위 동일 결과는 보장되지 않는다.

## 1. 설치

In [ ]:
# 이 셀이 하는 일: 생성 라이브러리 설치
!pip -q install -U diffusers transformers accelerate

## 2. 입력 포즈 만들기 (전신 스켈레톤 + 여백)
SD1.5는 다리를 잘라 그리는 버릇이 있어, **스켈레톤을 프레임 위쪽에 작게 배치(아래 여백 크게)** 해서 발이 프레임 안에 들어오게 한다.

In [ ]:
# 이 셀이 하는 일: 전신 스켈레톤을 여백 두고 캔버스에 배치 → 저장
import os
from PIL import Image
from diffusers.utils import load_image
os.makedirs('samples', exist_ok=True)
sk = load_image('https://huggingface.co/datasets/hf-internal-testing/diffusers-images/resolve/main/sd_controlnet/pose.png').convert('RGB')
W, H = 512, 768
fig = sk.resize((int(W*0.52), int(H*0.66)))
pose = Image.new('RGB', (W, H), 'black')
pose.paste(fig, ((W-fig.width)//2, int(H*0.05)))
pose.save('samples/pose_01.png')
pose

## 3. SD1.5 + ControlNet(OpenPose) 로드

In [ ]:
# 이 셀이 하는 일: 파이프라인 로드 (GPU면 cuda/float16, 아니면 cpu/float32)
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32
controlnet = ControlNetModel.from_pretrained('lllyasviel/sd-controlnet-openpose', torch_dtype=dtype)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5', controlnet=controlnet, torch_dtype=dtype, safety_checker=None)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to(device)
print('ready on', device)

## 4. 생성 함수 (고정 시드로 반복성 향상)

In [ ]:
# 이 셀이 하는 일: (프롬프트) -> 전신 이미지. 다리 살리는 설정 포함
neg = 'cropped, cut off at waist, cut off at knees, missing feet, no shoes, close-up, portrait, headshot, out of frame, lowres, bad anatomy, worst quality, blurry'
def gen(prompt, seed=42):
    g = torch.Generator(device).manual_seed(seed)
    return pipe(prompt, image=pose, height=H, width=W, num_inference_steps=20,
                controlnet_conditioning_scale=1.15, generator=g, negative_prompt=neg).images[0]

## 5. 실행 — 같은 포즈, 다른 프롬프트

In [ ]:
# 이 셀이 하는 일: 캐주얼 남자 생성 (전신)
out1 = gen('full length wide shot photo of a man standing, entire body from head to shoes, wearing sneakers, feet on the floor, casual clothes, plain studio background, highly detailed')
out1.save('samples/output_01.png')
out1

In [ ]:
# 이 셀이 하는 일: 정장 남자 생성 (같은 포즈, 옷만 변경)
out2 = gen('full length wide shot photo of a man standing, entire body from head to shoes, wearing dress shoes, feet on the floor, business suit, plain studio background, highly detailed')
out2.save('samples/output_02.png')
out2

## 6. 관찰 정리
- **같은 포즈 + 프롬프트만 변경:** 두 결과 모두 머리부터 발끝까지 **전신**, 같은 서있는 자세. 옷차림만 캐주얼↔정장으로 바뀜 → 입력 포즈 반영.
- **재현성:** seed=42를 고정해 같은 실행 환경에서 결과의 반복성을 높임. CPU/GPU, 연산 정밀도, 라이브러리 버전이 다르면 픽셀 단위 동일 결과는 보장되지 않음.
- **다리 잘림 해결:** 세로 프레임(768) + 스켈레톤 위쪽 배치(아래 여백) + conditioning_scale 1.15 + 'head to shoes' 프롬프트.